# Scratch Book

In [22]:
### Goals allowed over expected goals
import os
import sys
from pathlib import Path
import pandas as pd
import regex as re

import numpy as np
import requests
from bs4 import BeautifulSoup
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.ticker import FuncFormatter
import seaborn as sns
import matplotlib.font_manager as fm
from matplotlib.font_manager import FontProperties
from matplotlib.offsetbox import OffsetImage
from matplotlib.ticker import PercentFormatter
from matplotlib.ticker import ScalarFormatter
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from PIL import Image

# ======= BASE PATHS =======
try:
    # Works when running as a script
    base_dir = Path(__file__).resolve().parent
except NameError:
    # Fallback for notebooks or interactive mode
    base_dir = Path.cwd()

project_root = base_dir.parent


# ======= DATA FOLDERS =======
temp_folder = project_root / "TEMP"
data_folder = project_root / "data"
roster_folder = data_folder / "player_info"
school_info_folder = data_folder / "school_info"

# ======= IMAGE FOLDERS =======
img_folder = project_root / "images"
logo_folder = img_folder / "logos"
background_folder = img_folder / "background"
plot_folder = project_root / "TEMP" / "IMAGE" / "scoring_origins"

# ======= IMPORT CONFIG =======
import config  # now you can import config.py


school_info_file = school_info_folder / "arena_school_info.csv"
school_info_df = pd.read_csv(school_info_file)

# Check the Config import
# print((config_folder / "config.py").read_text())

In [23]:
## Connect to DB with 2025-26 data
import sqlalchemy
from sqlalchemy import create_engine

db_path = data_folder / "db" / "Season_YTD.db"
engine = create_engine(f"sqlite:///{db_path}")
connection = engine.connect()

# print list of tables in the database
inspector = sqlalchemy.inspect(engine)
tables = inspector.get_table_names()
print(tables)

# # extract Master Roster and save as a csv ### NOT USING THE ONE IN DB TRYING ONE FROM MARCH SCRAPE
# master_roster_df = pd.read_sql("SELECT * FROM master_roster", connection)
# master_roster_file = roster_folder / "roster_2024_25_new_scrape.csv"
# master_roster_df.to_csv(master_roster_file, index=False)

### READ master_roster from NEW SCRAPE CSV
master_roster_file = roster_folder / "roster_2024_25_new_scrape.csv"
master_roster_df = pd.read_csv(master_roster_file)

['advanced_metrics', 'game_details', 'goalie_stats', 'line_chart', 'linescore', 'penalty_summary', 'player_stats', 'player_stats_ytd', 'scoring_summary', 'shot_events']


In [24]:
## Get goals scored/allowed and xG for each team each game from the linescore table
linescore_df = pd.read_sql("SELECT * FROM linescore", connection)
print(linescore_df.head())

               Team  goals1  goals2  goals3  goalsT  shots1  shots2  shots3  \
0          Clarkson       0       0       1       1      10       8      14   
1          Canisius       1       1       1       3      10      14       8   
2       Connecticut       1       1       0       2       5      13      11   
3  Colorado College       1       2       1       4      16      13       9   
4     Bemidji State       1       5       3       9      12      11       8   

   shotsT  Pen  ...  FOL      FOW_1   xG  goals4  goals5  goals6  shots4  \
0      32    3  ...   33  52.857143  2.4       0       0       0       0   
1      32    2  ...   37  47.142857  1.9       0       0       0       0   
2      29    5  ...   32  50.000000  3.1       0       0       0       0   
3      38    5  ...   32  50.000000  3.6       0       0       0       0   
4      31    6  ...   37  47.142857  2.2       0       0       0       0   

   shots5  shots6                                    Game_ID  
0    

In [25]:
# load penalty summary data table to DF
penalty_summary_df = pd.read_sql("SELECT * FROM penalty_summary", connection)

print(penalty_summary_df.head())

       Period      Team            Player Pen_Length    Penalty_Type   Time  \
0  1st Period  Clarkson  Tristan Sarsland          2         Holding  11:36   
1  2nd Period  Clarkson    Ty Brassington          2  Cross-Checking   3:27   
2  2nd Period  Canisius   Rhys Chiddenton          2         Holding   7:09   
3  3rd Period  Clarkson      Bryce Sookro          2        Slashing   6:30   
4  3rd Period  Canisius   Rhys Chiddenton          2   High-sticking   7:22   

                        Game_ID  
0  2025-10-03-Clarkson-Canisius  
1  2025-10-03-Clarkson-Canisius  
2  2025-10-03-Clarkson-Canisius  
3  2025-10-03-Clarkson-Canisius  
4  2025-10-03-Clarkson-Canisius  


In [26]:
df = penalty_summary_df.copy()

# Map period labels to numbers
period_map = {
    "1st Period": 1,
    "2nd Period": 2,
    "3rd Period": 3
}

df["Period_Num"] = df["Period"].map(period_map)


In [27]:
df["Game_Date"] = pd.to_datetime(df["Game_ID"].str.slice(0, 10))


In [28]:
df["Game_Period_ID"] = (
    df["Game_Date"].astype(str)
    + "_"
    + df["Game_ID"]
    + "_P"
    + df["Period_Num"].astype(str)
)


In [29]:
df.head()

,Period,Team,Player,Pen_Length,Penalty_Type,Time,Game_ID,Period_Num,Game_Date,Game_Period_ID
0,1st Period,Clarkson,Tristan Sarsland,2,Holding,11:36,2025-10-03-Clarkson-Canisius,1.0,2025-10-03,2025-10-03_2025-10-03-Clarkson-Canisius_P1.0
1,2nd Period,Clarkson,Ty Brassington,2,Cross-Checking,3:27,2025-10-03-Clarkson-Canisius,2.0,2025-10-03,2025-10-03_2025-10-03-Clarkson-Canisius_P2.0
2,2nd Period,Canisius,Rhys Chiddenton,2,Holding,7:09,2025-10-03-Clarkson-Canisius,2.0,2025-10-03,2025-10-03_2025-10-03-Clarkson-Canisius_P2.0
3,3rd Period,Clarkson,Bryce Sookro,2,Slashing,6:30,2025-10-03-Clarkson-Canisius,3.0,2025-10-03,2025-10-03_2025-10-03-Clarkson-Canisius_P3.0
4,3rd Period,Canisius,Rhys Chiddenton,2,High-sticking,7:22,2025-10-03-Clarkson-Canisius,3.0,2025-10-03,2025-10-03_2025-10-03-Clarkson-Canisius_P3.0


In [30]:
teams = df["Team"].unique()
games = df[["Game_ID", "Game_Date"]].drop_duplicates()

periods = pd.DataFrame({"Period_Num": [1, 2, 3]})

full_index = (
    games
    .merge(periods, how="cross")
    .merge(pd.DataFrame({"Team": teams}), how="cross")
)


In [31]:
penalties = (
    df.groupby(["Game_ID", "Period_Num", "Team"])
      .size()
      .reset_index(name="Penalty_Count")
)

full = full_index.merge(
    penalties,
    on=["Game_ID", "Period_Num", "Team"],
    how="left"
)

full["Penalty_Count"] = full["Penalty_Count"].fillna(0)
full["No_Penalty"] = full["Penalty_Count"] == 0




In [32]:
full = full.sort_values(
    ["Team", "Game_Date", "Game_ID", "Period_Num"]
)


In [33]:
full["Penalty_Break"] = ~full["No_Penalty"]

full["Streak_ID"] = (
    full.groupby("Team")["Penalty_Break"]
        .cumsum()
)

streaks = (
    full[full["No_Penalty"]]
    .groupby(["Team", "Streak_ID"])
    .size()
    .reset_index(name="Periods_Without_Penalty")
)


full.head()

,Game_ID,Game_Date,Period_Num,Team,Penalty_Count,No_Penalty,Penalty_Break,Streak_ID
425,2025-10-03-Bemidji State-Alaska Anchorage,2025-10-03,1,Air Force,0.0,True,False,0
488,2025-10-03-Bemidji State-Alaska Anchorage,2025-10-03,2,Air Force,0.0,True,False,0
551,2025-10-03-Bemidji State-Alaska Anchorage,2025-10-03,3,Air Force,0.0,True,False,0
47,2025-10-03-Clarkson-Canisius,2025-10-03,1,Air Force,0.0,True,False,0
110,2025-10-03-Clarkson-Canisius,2025-10-03,2,Air Force,0.0,True,False,0


In [34]:
streaks.head()

,Team,Streak_ID,Periods_Without_Penalty
0,Air Force,0,123
1,Air Force,1,1
2,Air Force,2,69
3,Air Force,3,1
4,Air Force,4,102


In [12]:
teams_4_plus = streaks[streaks["Periods_Without_Penalty"] >= 4]

teams_4_plus.sort_values(
    "Periods_Without_Penalty",
    ascending=False
)


,Team,Streak_ID,Periods_Without_Penalty
451,Harvard,0,696
1335,Yale,0,690
385,Dartmouth,0,640
1008,Penn State,43,576
369,Cornell,0,558
...,...,...,...
323,Colgate,46,6
675,Miami,47,6
165,Bentley,19,5
461,Harvard,22,4


In [ ]:
breakpoint

In [ ]:
import pandas as pd

def build_goals_xgoals_shots_comparison(linescore_df: pd.DataFrame) -> pd.DataFrame:
    """
    Transform a linescore_df (2 rows per Game_ID) into:

    Game_ID, Team, goals_scored, xgoals_scored,
             goals_allowed, xgoals_allowed,
             shots_taken, shots_allowed
    """

    df = linescore_df.copy()

    # Ensure numeric types for safety
    df["goalsT"] = pd.to_numeric(df["goalsT"], errors="coerce")
    df["xG"] = pd.to_numeric(df["xG"], errors="coerce")
    df["shotsT"] = pd.to_numeric(df["shotsT"], errors="coerce")

    # Rename for clarity before merging
    df = df.rename(columns={
        "goalsT": "goals_scored",
        "xG": "xgoals_scored",
        "shotsT": "shots_taken",
    })

    # Self-merge on Game_ID to pair each team with its opponent
    merged = df.merge(
        df,
        on="Game_ID",
        suffixes=("", "_opp")
    )

    # Drop self-matches (Team vs itself)
    merged = merged[merged["Team"] != merged["Team_opp"]].copy()

    # Select and rename columns into the final structure
    final = merged[[
        "Game_ID",
        "Team",
        "goals_scored",
        "xgoals_scored",
        "shots_taken",
        "goals_scored_opp",
        "xgoals_scored_opp",
        "shots_taken_opp",
    ]].rename(columns={
        "goals_scored_opp": "goals_allowed",
        "xgoals_scored_opp": "xgoals_allowed",
        "shots_taken_opp": "shots_allowed",
    })

    # Optional: sort for sanity
    final = final.sort_values(["Game_ID", "Team"], ignore_index=True)

    return final
goals_xgoals_comparison_df = build_goals_xgoals_shots_comparison(linescore_df)
print(goals_xgoals_comparison_df.head(10))

In [ ]:
def aggregate_team_season_stats(team_game_df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregates game-by-game stats into season totals and per-game averages
    for each team.
    """

    df = team_game_df.copy()

    # Group by Team and aggregate sums + counts
    grouped = df.groupby("Team").agg(
        games_played=("Game_ID", "nunique"),
        goals_scored=("goals_scored", "sum"),
        xgoals_scored=("xgoals_scored", "sum"),
        goals_allowed=("goals_allowed", "sum"),
        xgoals_allowed=("xgoals_allowed", "sum"),
        shots_taken=("shots_taken", "sum"),
        shots_allowed=("shots_allowed", "sum"),
    )

    # Per-game averages
    grouped["goals_scored_per_game"] = grouped["goals_scored"] / grouped["games_played"]
    grouped["xgoals_scored_per_game"] = grouped["xgoals_scored"] / grouped["games_played"]
    grouped["goals_allowed_per_game"] = grouped["goals_allowed"] / grouped["games_played"]
    grouped["xgoals_allowed_per_game"] = grouped["xgoals_allowed"] / grouped["games_played"]
    grouped["shots_taken_per_game"] = grouped["shots_taken"] / grouped["games_played"]
    grouped["shots_allowed_per_game"] = grouped["shots_allowed"] / grouped["games_played"]

    return grouped.reset_index()
team_season_stats_df = aggregate_team_season_stats(goals_xgoals_comparison_df)
print(team_season_stats_df.head(10))

In [ ]:
### Shots taken and allowed per expected goals scored/allowed

team_season_stats_df['shots_per_xgoals_scored'] = team_season_stats_df['shots_taken'] / team_season_stats_df['xgoals_scored']
team_season_stats_df['shots_per_xgoals_allowed'] = team_season_stats_df['shots_allowed'] / team_season_stats_df['xgoals_allowed']

In [ ]:
team_season_stats_df

In [ ]:
break

## Scoring by origin for 2024-25 full season stats

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import regex as re

import numpy as np
import requests
from bs4 import BeautifulSoup
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.ticker import FuncFormatter
import seaborn as sns
import matplotlib.font_manager as fm
from matplotlib.font_manager import FontProperties
from matplotlib.offsetbox import OffsetImage
from matplotlib.ticker import PercentFormatter
from matplotlib.ticker import ScalarFormatter
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from PIL import Image

# ======= BASE PATHS =======
try:
    # Works when running as a script
    base_dir = Path(__file__).resolve().parent
except NameError:
    # Fallback for notebooks or interactive mode
    base_dir = Path.cwd()

project_root = base_dir.parent


# ======= DATA FOLDERS =======
temp_folder = project_root / "TEMP"
data_folder = project_root / "data"
roster_folder = data_folder / "player_info"
school_info_folder = data_folder / "school_info"

# ======= IMAGE FOLDERS =======
img_folder = project_root / "images"
logo_folder = img_folder / "logos"
background_folder = img_folder / "background"
plot_folder = project_root / "TEMP" / "IMAGE" / "scoring_origins"

# ======= IMPORT CONFIG =======
import config  # now you can import config.py


school_info_file = school_info_folder / "arena_school_info.csv"
school_info_df = pd.read_csv(school_info_file)

# Check the Config import
# print((config_folder / "config.py").read_text())

In [ ]:
## Get New Copy of 2024_25 Roster using old scraper



In [ ]:
## Connect to DB with 2024-25 data
import sqlalchemy
from sqlalchemy import create_engine

db_path = data_folder / "db" / "2024_2025_Season_Final.db"
engine = create_engine(f"sqlite:///{db_path}")
connection = engine.connect()

# print list of tables in the database
inspector = sqlalchemy.inspect(engine)
tables = inspector.get_table_names()
print(tables)

# # extract Master Roster and save as a csv ### NOT USING THE ONE IN DB TRYING ONE FROM MARCH SCRAPE
# master_roster_df = pd.read_sql("SELECT * FROM master_roster", connection)
# master_roster_file = roster_folder / "roster_2024_25_new_scrape.csv"
# master_roster_df.to_csv(master_roster_file, index=False)

### READ master_roster from NEW SCRAPE CSV
master_roster_file = roster_folder / "roster_2024_25_new_scrape.csv"
master_roster_df = pd.read_csv(master_roster_file)

In [ ]:
# get ytd_stats table
ytd_stats_df = pd.read_sql("SELECT * FROM player_stats_ytd", connection)
ytd_stats_file = roster_folder / "2024_25_Season Stats.csv"
ytd_stats_df.to_csv(ytd_stats_file, index=False)

## Find a player on rensselaer, last name Caron. find player in ytd_stats_df that contains 'Caron' in Clean_Player column
caron_player = ytd_stats_df[ytd_stats_df['Clean_Player'].str.contains('Caron', na=False)]

## Simplify player name from Félix Caron to Felix Caron for matching
caron_player['Clean_Player'] = caron_player['Clean_Player'].str.replace('é', 'e')

print(caron_player)

In [ ]:
## # Combine First and last name in roster_df to match player_ytd_df
roster_df = master_roster_df
# Clean white space from names
roster_df["First_Name"] = roster_df["First_Name"].str.strip()
roster_df["Last_Name"] = roster_df["Last_Name"].str.strip()
roster_df["Clean_Player"] = roster_df["First_Name"] + " " + roster_df["Last_Name"]
# Strip any leading/trailing whitespace
roster_df["Clean_Player"] = roster_df["Clean_Player"].str.strip()
# Rename Current Team to match player_ytd_df
roster_df = roster_df.rename(columns={"Current Team": "Team"})
# Reorder columns for easier viewing
#Order of columns
# ["No","Team","Clean_Player", "First_Name","Last_Name", 'Position', 'Yr', 'Ht', 'Wt', 'DOB', 'Hometown', 'Height_Inches', 'Draft_Year', 'NHL_Team', 'D_Round', 'Last Team', 'League', 'City', 'State_Province', 'Country']
roster_df = roster_df[["No","Team","Clean_Player", "First_Name","Last_Name", 'Position', 'Yr', 'Ht', 'Wt', 'DOB', 'Hometown', 'Height_Inches', 'Draft_Year', 'NHL_Team', 'D_Round', 'Last Team', 'League', 'City', 'State_Province', 'Country']]    




roster_df.head()

## State_Province Value Counts
check_states = roster_df["State_Province"].value_counts(dropna=False)
check_states


### Roster Fixes and Mods

In [ ]:
### Roster Modifications to make up for incomplete data ###

### Replace bad values in State_Province column
state_province_corrections = {"Okla.": "Oklahoma", "D.C": "District of Columbia"
}
# Apply state/province corrections
roster_df["State_Province"] = roster_df["State_Province"].replace(state_province_corrections)

### Replace country abbreviations with full country names
country_corrections = {"CYM": "Cayman Islands", "JPN": "Japan", "RUS": "Russia"
}
# Apply country corrections
roster_df["Country"] = roster_df["Country"].replace(country_corrections)

In [ ]:
player_ytd_df = ytd_stats_df

# Make sure name and team columns are stripped of punctuation, strange characters, and whitespace
player_ytd_df["Clean_Player"] = player_ytd_df["Clean_Player"].str.strip()
player_ytd_df["Team"] = player_ytd_df["Team"].str.strip()
roster_df["Clean_Player"] = roster_df["Clean_Player"].str.strip()
roster_df['Clean_Player'] = roster_df['Clean_Player'].str.replace(u'\xa0', u' ').str.strip()  # Cleanup player name
roster_df["Team"] = roster_df["Team"].str.strip()
# # Change American Int'l to American Intl in Team Columns
player_ytd_df["Team"] = player_ytd_df["Team"].replace("American Int'l", "American Intl")
roster_df["Team"] = roster_df["Team"].replace("American Int'l", "American Intl")
# Remove any hyphens & periods from team names, replace with ' '
player_ytd_df["Team"] = player_ytd_df["Team"].str.replace(r'[^\w\s]', ' ', regex=True)
roster_df["Team"] = roster_df["Team"].str.replace(r'[^\w\s]', ' ', regex=True)
# Remove any hyphens, periods, ect from Clean Player names to match
# player_ytd_df["Clean_Player"] = player_ytd_df["Clean_Player"].str.replace(r'[^\w\s]', '', regex=True)
# roster_df["Clean_Player"] = roster_df["Clean_Player"].str.replace(r'[^\w\s]', '', regex=True)
# QUICK FIX - Standardize team names with double spaces
# If team name column has double spaces, replace with single space
player_ytd_df["Team"] = player_ytd_df["Team"].str.replace('  ', ' ', regex=False)
roster_df["Team"] = roster_df["Team"].str.replace('  ', ' ', regex=False)

## Merge the two DataFrames on Clean_Player and Team
merged_df = pd.merge(
    player_ytd_df,
    roster_df,
    left_on=["Clean_Player", "Team"],
    right_on=["Clean_Player", "Team"],
    how="left"
)

# Shape and info of merged DataFrame
print("Merged DataFrame shape:", merged_df.shape)
# print(merged_df.columns)
# print(merged_df.head())
# merged_df.info()



### Insert Last Team and League from manually researched ones

In [ ]:
## Load csv file with player origins needed in corrections
corrections_file = roster_folder / "2024_25_player_origin_corrections.csv"
corrections_df = pd.read_csv(corrections_file)

corrections_df.head()

# Use Clean_Player as key, create dictionary with Last_Team and League
corrections = {}
for index, row in corrections_df.iterrows():
    corrections[row['Clean_Player']] = {
        'Last_Team': row['Last Team'],
        'League': row['League']
    }

print(corrections)

## Apply corrections to roster_df
def apply_corrections(row):
    player_name = row['Clean_Player']
    if player_name in corrections:
        row['Last Team'] = corrections[player_name]['Last_Team']
        row['League'] = corrections[player_name]['League']
    return row

merged_df = merged_df.apply(apply_corrections, axis=1)

In [ ]:
merged_df.head()

In [ ]:
### Print dataframe stats for to keep track of filtering steps
original_count = merged_df.shape[0]
print("Merged DataFrame shape before filtering:", merged_df.shape)

### Filter out players with no TOI and goalies
# Remove players that haven't appeared in a game at all this year (TOI_sec = 0)
# Remove any Rows where TOI_sec is 0 or NaN - These are goalies or players with no time on ice
merged_df = merged_df[(merged_df["TOI_sec"] > 0) & (~merged_df["TOI_sec"].isna())]
first_step_count = merged_df.shape[0]

## Check the shape after filtering
print("Merged DataFrame shape after filtering TOI_sec > 0:", merged_df.shape)
# Number of players removed
print("Players removed after filtering TOI_sec > 0:", original_count - first_step_count)

### DO NOT NEED TO FILTER FOR GOALIES BECAUSE PLAYER_YTD_STATS TABLE ONLY HAS TOI FOR SKATERS
# Filter out goaltenders as non skaters - looking at offensive production so goalies are irrelivant (despite the assists they may get from time to time)
# Strip any whitespace from Position column
# merged_df["Position"] = merged_df["Position"].str.strip()
# merged_df = merged_df[merged_df["Position"] != "Goaltenders"]
# no_goalie_count = merged_df.shape[0]
## Check the shape after filtering
# print("Merged DataFrame shape after filtering out Goalies:", merged_df.shape)
# # Number of players removed
# print("Players removed by filtering out Goalies:", first_step_count - no_goalie_count)

In [ ]:
### Reusing League and Team Classification function and libraries from team_construction_visual_workbook import classify_previous_team, classify_previous_league

# ----------------------------
# 1) Rename merged_df to df for easier to fit in with existing code
# -----------------------------
df = merged_df.copy()

# -----------------------------
# 2) Classification helpers
# -----------------------------
def _norm_set(strings):
    return { _norm(s) for s in strings }

def _norm(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).strip().upper()
    s = re.sub(r"[.\u2010-\u2015\-–—]+", " ", s)  # unify hyphen-like chars to space
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _norm_set(strings):
    return { _norm(s) for s in strings }

NTDP_TEAM_HINTS_RAW = (
    "USA U 18", "US U 18", "USA U18", "US U18",
    "USA U 17", "US U 17", "USA U17", "US U17",
    "NTDP", "USNTDP", "US NATIONAL TEAM", "US DEV PROGRAM", "US DEVELOPMENT PROGRAM"
)
NTDP_LEAGUE_HINTS_RAW = ("NTDP", "USNTDP")

D1_CONFS_RAW = {"ECAC","CCHA","NCHC","HEA","B10","AHA","INDEPENDENTS","D I IND","NCAA", "NCAA-DI"}

CJHL_LEAGUES_RAW = {"BCHL","AJHL","SJHL","OJHL", "MJHL", "CCHL","MHL"}

US_TIER1_RAW = {"USHL"}
US_TIER2_RAW = {"NAHL","NCDC"}
US_OTHER_RAW  = {"USPHL","NA3HL", "PREP", "PHC", "USHS", "CISAA", "DIII", "D-III", "EHL", "ACHA"}

CHL_RAW = {"OHL","WHL","QMJHL"}

EURO_HINTS_RAW = {
    "J20 NATIONELL","J18 REGION","U20 SM SARJA","U18","U20","SM SARJA",
    "SHL","ICEHL","ALPSHL","LIIGA","MHL RUSSIA","KHL, ICEHL","KHL","DEL", "EC-KAC", 
    "Oberliga", "Europe", "SWE", "Sweden"
}

EURO_TEAM_HINTS_RAW = {
    "KalPa U20", "Frölunda HC", "Malmo", "Jokerit U20", "U20 SM Sarja-Pelicans",
    "Tappara J20", "Djurgårdens IF", "Leksands IF", "Mora IK J20"
}

RUSSIAN_MHL_TEAM_HINTS_RAW = (
    "KRASNAYA","LOKO","MOSKVA","MOSCOW","ST PETERSBURG","SKA","LOKOMOTIV",
    "DMITROV","CHELYABINSK","OMSK","NOVOSIBIRSK","MAGNITOGORSK","NIZHNY","YAROSLAVL",
    "Karlskrona HK"
)

# Normalize them
NTDP_TEAM_HINTS       = _norm_set(NTDP_TEAM_HINTS_RAW)
NTDP_LEAGUE_HINTS     = _norm_set(NTDP_LEAGUE_HINTS_RAW)
D1_CONFS              = _norm_set(D1_CONFS_RAW)
CJHL_LEAGUES          = _norm_set(CJHL_LEAGUES_RAW)
US_TIER1              = _norm_set(US_TIER1_RAW)
US_TIER2              = _norm_set(US_TIER2_RAW)
US_OTHER              = _norm_set(US_OTHER_RAW)
CHL                   = _norm_set(CHL_RAW)
EURO_HINTS            = _norm_set(EURO_HINTS_RAW)
EURO_TEAM_HINTS       = _norm_set(EURO_TEAM_HINTS_RAW)
RUSSIAN_MHL_TEAM_HINTS = _norm_set(RUSSIAN_MHL_TEAM_HINTS_RAW)

PRO_HINTS = ("AHL","ECHL")

BIN_ORDER = [
    "NTDP",
    "USHL (non‑NTDP)",
    "NAHL/NCDC",
    "US (DIII/Prep/Other)",
    "CHL (Major Junior)",
    "CJHL (Canadian Jr A)",
    "U SPORTS",
    "Europe",
    "NCAA D1 Transfers",
    "Pro (AHL/ECHL/Other)",
    "Other/Various/Unknown"
    
]

COLOR_MAP = {
    "NTDP": "#0057B8",
    "USHL (non‑NTDP)": "#1E90FF",
    "NAHL/NCDC": "#63B8FF",
    "US (DIII/Prep/Other)": "#B0E2FF",
    "CHL (Major Junior)": "#B22222",
    "CJHL (Canadian Jr A)": "#FF7F7F",
    "U SPORTS": "#FADBD8",
    "Europe": "#F0E130",
    "NCAA D1 Transfers": "#696969",
    "Pro (AHL/ECHL/Other)": "#000000",
    "Other/Various/Unknown": "#A9A9A9"
}

def classify_prev_bin(last_team: str, league: str) -> str:
    t = _norm(last_team)
    l = _norm(league)

    # NTDP carve-out first
    if any(h in t for h in NTDP_TEAM_HINTS) or any(h == l for h in NTDP_LEAGUE_HINTS):
        return "NTDP"

    # D3/Prep -> Other
    if l in US_OTHER:
        return "US (DIII/Prep/Other)"

    # NCAA D1 transfers
    if (l in D1_CONFS) or ("NCAA" in l and l != ""):
        return "NCAA D1 Transfers"

    # U SPORTS
    if l in {"USPORTS", "U SPORTS"}:
        return "U SPORTS"

    # CHL
    if l in CHL:
        return "CHL (Major Junior)"

    # MHL ambiguity
    if l == "MHL":
        if any(k in t for k in RUSSIAN_MHL_TEAM_HINTS):
            return "Europe"
        else:
            return "CJHL (Canadian Jr A)"

    # CJHL
    if l in CJHL_LEAGUES:
        return "CJHL (Canadian Jr A)"



    # US juniors
    if l in US_TIER1:
        return "USHL (non‑NTDP)"
    if l in US_TIER2:
        return "NAHL/NCDC"
    
    # Pro leagues
    if any(h in l for h in PRO_HINTS):
        return "Pro (AHL/ECHL/Other)"

    # Europe consolidated
    if l in EURO_HINTS:
        return "Europe"
    if t in EURO_TEAM_HINTS:
        return "Europe"




    # Fallbacks/Unknown
    # if l == "" or pd.isna(league):
    #     return "Other/Various/Unknown"
    
    if "U SPORTS" in t:
        return "U SPORTS"
    if "IF Sundsvall" in t:  # edge case
        return "Europe"


    return "Other/Various/Unknown"


# Apply classification to the raw roster
df["Prev_League_Bin"] = df.apply(lambda r: classify_prev_bin(r.get("Last Team", np.nan),
                                                             r.get("League", np.nan)), axis=1)
df["Prev_League_Bin"] = pd.Categorical(df["Prev_League_Bin"], categories=BIN_ORDER, ordered=True)

# Save classified to temp_folder
# timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# out_file = os.path.join(temp_folder, f"roster_classified_{timestamp}.csv")
# df.to_csv(out_file, index=False)


# show quick counts
overall_counts = (df["Prev_League_Bin"]
                  .value_counts(dropna=False)
                  .reindex(BIN_ORDER)
                  .fillna(0).astype(int))

print("Overall Counts by Prev_League_Bin")
print(overall_counts.reset_index().rename(columns={"index":"Bin","Prev_League_Bin":"Count"}))

In [ ]:
## Show The Other/Various/Unknown players
other_unknown_df = df[df["Prev_League_Bin"] == "Other/Various/Unknown"]
other_unknown_df = other_unknown_df[["Clean_Player", "Team", "Last Team", "League", "Prev_League_Bin" ]]
other_unknown_df

In [ ]:
### Output 2024_25 merged_df with classifications to csv
output_file = roster_folder / "2024_25_Player_Stats_with_Origins_NEW.csv"
df.to_csv(output_file, index=False)

### Create Library of Color Map and logos

In [ ]:
# # create library of team color mappings and logos to use in plots

# import os
# import pandas as pd

# # path to TEMP folder
# temp_folder = os.path.join(os.getcwd(), '..', 'TEMP')
# # Data folder
# data_folder = os.path.join(os.getcwd(), '..', 'data')
# # print(os.listdir(data_folder)) # Print List of files in data folder

# # path to School Info folder in data
# school_info_folder = os.path.join(os.getcwd(), data_folder, 'school_info')

# # Image folders
# img_folder = os.path.join(os.getcwd(), '..', '..', 'images') # base image folder
# # Logo folder
# logo_folder = os.path.join(os.getcwd(), '..', '..', 'images', 'logos')
# # print(os.listdir(logo_folder)) # Print List of files in logo folder

# # Background folder
# background_folder = os.path.join(img_folder, 'background')
# # print(os.listdir(background_folder)) # Print List of files in background folder

# # Plot Output folder
# plot_folder = os.path.join(os.getcwd(), '..', '..', 'TEMP', 'IMAGE')



# ################################################################################


# # Path to school info table (csv)
# school_info_file = os.path.join(school_info_folder, 'arena_school_info.csv')
# school_info_df = pd.read_csv(school_info_file)

In [ ]:
school_info_df

In [ ]:
# # Preprocess school info dataframe to create color and logo mappings
# # hex1 and hex2 columns need to be normalized to ensure they are 6-character hex codes
# def normalize_hex_color(hex_color):
#     if pd.isna(hex_color):
#         return None
#     hex_color = str(hex_color).lstrip('#')
#     if len(hex_color) == 3:
#         hex_color = ''.join([c*2 for c in hex_color])
#     return f'#{hex_color.zfill(6)}'
# school_info_df['hex1'] = school_info_df['hex1'].apply(normalize_hex_color)
# school_info_df['hex2'] = school_info_df['hex2'].apply(normalize_hex_color)
# school_info_df['hex3'] = school_info_df['hex3'].apply(normalize_hex_color)

# # Create color mapping dictionary
# team_color_map = {}
# for _, row in school_info_df.iterrows():
#     team_name = row['Team']
#     color1 = row['hex1'] if pd.notna(row['hex1']) else '#000000'  # default to black
#     color2 = row['hex2'] if pd.notna(row['hex2']) else '#FFFFFF'  # default to white
#     color3 = row['hex3'] if pd.notna(row['hex3']) else '#CCCCCC'  # default to gray
#     team_color_map[team_name] = (color1, color2, color3)

# # Create logo mapping dictionary
# team_logo_map = {}
# for _, row in school_info_df.iterrows():
#     team_name = row['Team']
#     logo_path = row['logo_abv'] if pd.notna(row['logo_abv']) else None
#     team_logo_map[team_name] = logo_path


In [ ]:
team_color_map